# CTNSG Master Kaggle Training Notebook
This notebook orchestrates the end-to-end training pipeline for the Canonical Tractable Neuro-Symbolic Generation Framework.
It assumes execution on Kaggle with dual T4 or P100 GPUs (13-16GB VRAM).

**Note:** This notebook is configured for full training.

In [ ]:
!git clone https://github.com/Borisz42/CTNSG.git
%cd CTNSG
%pip install -r requirements.txt

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim

# Ensure local modules are reachable
sys.path.append(os.path.abspath('.'))

from macroplanner.gvt.model import GraphVQTransformer
from macroplanner.reldit.model import RelDiT
from orchestrator.arbor.planner import ArborPlanner
from realizer.realizer import CTNSGRealizer
from contracts.graph_schema import DiscourseGraph, SemanticNode, SemanticEdge

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on {device}")

## 1. Load Preprocessed Supervisor Datasets (Module 2 & 4)
Loading the true DAGs, SDRT discourse trees, FAAP instruction pairs, and SAIGuard interaction topologies.

In [ ]:
import json
import os
from huggingface_hub import hf_hub_download

def fetch_and_load(filename):
    local_path = f'processed_data/{filename}'
    if not os.path.exists(local_path):
        print(f'Downloading {filename} from Hugging Face...')
        try:
            os.makedirs('processed_data', exist_ok=True)
            downloaded = hf_hub_download(repo_id='Borisz42/CTNSG-Graph-Curriculum', repo_type='dataset', filename=filename)
            import shutil
            shutil.copy(downloaded, local_path)
        except Exception as e:
            print(f'Warning: Could not download {filename}. {e}')
    
    if filename.endswith('.pt'):
        return torch.load(local_path) if os.path.exists(local_path) else []
    elif filename.endswith('.jsonl'):
        data = []
        if os.path.exists(local_path):
            with open(local_path, 'r', encoding='utf-8') as f:
                data = [json.loads(line) for line in f]
        return data

# Load FAAP (Seq2Seq)
faap_data = fetch_and_load('faap_instructions_full.jsonl')
print(f"Loaded {len(faap_data)} FAAP instruction pairs.")

# Load Topological Graphs
curriculum_graphs = fetch_and_load('ctnsg_curriculum.pt')
    sdrt_graphs = fetch_and_load('sdrt_graphs_full.pt')
arbor_graphs = fetch_and_load('arbor_graphs_full.pt')
verification_graphs = fetch_and_load('verification_graphs_full.pt')

print(f"Loaded {len(curriculum_graphs)} Curriculum Graphs.")
    print(f"Loaded {len(sdrt_graphs)} SDRT Discourse Graphs.")
print(f"Loaded {len(arbor_graphs)} Arbor TDP True DAGs.")
print(f"Loaded {len(verification_graphs)} SAIGuard/Brick Interaction Graphs.")

## 2. Initialize Global Orchestrator
Decoupling semantic and structural intent via Arbor.

In [ ]:
planner = ArborPlanner(input_dim=512, hidden_dim=256).to(device)
global_intent = torch.randn(1, 512).to(device)
decoupled_plan = planner.decouple_plan(global_intent)
print("Decoupled Plan Confidence:", decoupled_plan['confidence'])

## 3. GVT (Graph Vector Tokenization) Training Loop
Compressing continuous topologies into discrete graph tokens.

In [ ]:
gvt = GraphVQTransformer(in_channels=256, hidden_channels=256, num_embeddings=64, num_quantizers=4).to(device)
gvt_optimizer = optim.AdamW(gvt.parameters(), lr=3e-4)

print("Starting GVT Training Loop (Full Training)...")
for epoch in range(100):
    total_loss_epoch = 0.0
    for graph in curriculum_graphs:
        nodes = graph['nodes'].to(device)
        edges = graph['edges'].to(device)
        gvt_optimizer.zero_grad()
        out = gvt(nodes, edges)
        quantized_latents = out['z_q']
        vq_loss = out['commit_loss']
        
        recon_loss = nn.MSELoss()(quantized_latents, nodes)
        total_loss = recon_loss + vq_loss
        
        total_loss.backward()
        gvt_optimizer.step()
        total_loss_epoch += total_loss.item()
    
    print(f"Epoch {epoch+1} | GVT Loss: {total_loss_epoch / max(1, len(curriculum_graphs)):.4f}")

discrete_indices = out['discrete_tokens']
print("GVT Tokenization complete. Sample discrete indices shape:", discrete_indices.shape)

## 4. RelDiT (Relational Diffusion) Training Loop
Training the discrete diffusion model to generate topologies.

In [ ]:
# RelDiT acts on the discrete indices generated by GVT
reldit = RelDiT(vocab_size=64, d_model=256).to(device)
reldit_optimizer = optim.AdamW(reldit.parameters(), lr=1e-4)

print("\nStarting RelDiT Training Loop (Full Training)...")
for epoch in range(100):
    total_loss_epoch = 0.0
    for graph in curriculum_graphs:
        nodes = graph['nodes'].to(device)
        edges = graph['edges'].to(device)
        
        with torch.no_grad():
            out = gvt(nodes, edges)
            tokens = out['discrete_tokens'][:, 0].unsqueeze(0)
            
        reldit_optimizer.zero_grad()
        loss = reldit(tokens)
        loss.backward()
        reldit_optimizer.step()
        total_loss_epoch += loss.item()
        
    print(f"Epoch {epoch+1} | RelDiT Loss: {total_loss_epoch / max(1, len(curriculum_graphs)):.4f}")

## 5. Realizer Inference Pipeline
Integrating the VNPool, O(1) GREATGRAMMA enforcement, and SafeLLM to generate text.

In [ ]:
print("\nInitializing Realizer Pipeline...")
realizer = CTNSGRealizer()

# Create a stub discourse graph to feed into the Realizer
inference_graph = DiscourseGraph(
    graph_id="infer_001",
    nodes=[
        SemanticNode(node_id="n1", concept="System", vq_index=12),
        SemanticNode(node_id="n2", concept="Macroplanner", vq_index=45)
    ],
    edges=[
        SemanticEdge(source_id="n1", target_id="n2", relation_type="triggers")
    ]
)

schema = {"type": "object", "properties": {"output": {"type": "string"}}}
result = realizer.generate(inference_graph, schema, context_lines=5)

print("\n=== Realizer Final Output ===")
print(result)

## 6. Model Weight Export & Distribution
Saves the trained components (GVT and RelDiT) to /kaggle/working/ and packages them into a .zip file. This zip file can be easily downloaded or natively linked as a 'Notebook Output' dataset into the Evaluation Suite notebook without needing manual downloads.

In [ ]:
import shutil

print('\nSaving trained models...')
os.makedirs('ctnsg_export', exist_ok=True)
torch.save(gvt.state_dict(), 'ctnsg_export/gvt_weights.pt')
torch.save(reldit.state_dict(), 'ctnsg_export/reldit_weights.pt')

# Create an export manifest
manifest = {
    'architecture': 'CTNSG',
    'modules': ['GVT', 'RelDiT'],
    'dim': 256
}
with open('ctnsg_export/manifest.json', 'w') as f:
    json.dump(manifest, f, indent=4)

# Zip the export directory
print('Packaging into ctnsg_release.zip...')
shutil.make_archive('/kaggle/working/ctnsg_release', 'zip', 'ctnsg_export')

print('Export complete! ctnsg_release.zip is now available in the Kaggle /kaggle/working/ directory.')